# **Conditional chains**

In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")


from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser


# ============================================================
# LLM
# ============================================================

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)


# ============================================================
# PYDANTIC SCHEMA
# ============================================================

from pydantic import BaseModel
from typing import Literal


class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]


# Create structured output LLM
llm_structured_output = llm_openai.with_structured_output(llm_schema)


# ============================================================
# CONDITIONAL CHAIN
# ============================================================

# TASK 1 - Prompt

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    (
        "human",
        "Please categorize the movie review as positive or negative : {input}"
    )
])


# ============================================================
# TASK 2 - STRUCTURED OUTPUT LLM
# ============================================================

llm_structured_output = llm_openai.with_structured_output(llm_schema)


# ============================================================
# TASK 3 - CUSTOM RUNNABLE
# ============================================================

from langchain_core.runnables import RunnableLambda


def pydantic_json(input: llm_schema) -> str:

    return input.model_dump()["movie_summary_flag"]


pydantic_json_lambda = RunnableLambda(pydantic_json)


# ============================================================
# CONDITIONAL CHAIN 1 - LINKEDIN
# ============================================================

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    (
        "human",
        "Create a post for the following text for LinkedIn: {text}"
    )
])


# LLM

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)


# String parser

str_parser = StrOutputParser()


# LinkedIn chain

chain_linkedin = (
    linkedin_prompt
    | llm_openai
    | str_parser
)


# ============================================================
# CONDITIONAL CHAIN 2 - INSTAGRAM
# ============================================================

from langchain_core.runnables import (
    RunnableParallel,
    RunnableLambda,
    RunnableBranch
)


def insta_chain(text: dict):

    text = text["text"]

    # Prompt
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn post generator"),
        (
            "human",
            "Create a post for the following text for Instagram: {text}"
        )
    ])

    # LLM
    llm_openai = ChatOpenAI(
        model="gpt-5-mini",
        temperature=0
    )

    # String parser
    str_parser = StrOutputParser()

    # Instagram chain
    chain_insta = (
        insta_prompt
        | llm_openai
        | str_parser
    )

    result = chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)


# ============================================================
# FINAL ORCHESTRATION
# ============================================================

conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)


final_orchestrator = (
    prompt_template
    | llm_structured_output
    | pydantic_json_lambda
    | conditional_chain
)


# ============================================================
# INVOKE
# ============================================================

result = final_orchestrator.invoke({
    "input": "I loved this KGF movie"
})

print(result)